# Taller 2: Expedición Lunar — Cartas desde la Luna con LLMs

## Bienvenidos, astronautas

El año es **2042**. Ustedes forman parte de la tripulación **Artemis VII**, la primera misión de larga duración en la superficie lunar. Llevan 3 meses en la **Base Selene** y hoy, por fin, la ventana de comunicación con la Tierra se ha abierto.

Tienen una única oportunidad para enviar una carta a sus familias. Pero aquí está el problema: el sistema de comunicaciones de la base usa un **modelo de lenguaje (LLM)** como asistente de redacción. Dependiendo de cómo configuren sus parámetros, la carta será:

- Fría y precisa... o cálida y creativa
- Corta y directa... o larga y detallada
- Repetitiva y monótona... o variada y expresiva

Su misión en este taller: **dominar los parámetros del LLM para escribir la carta perfecta.**

### Lo que aprenderán hoy

| Parámetro | Qué controla | Equivalente en Ollama |
|---|---|---|
| `temperature` | Creatividad vs precisión | `temperature` |
| `top_p` | Diversidad del vocabulario | `top_p` |
| `max_tokens` | Longitud de la respuesta | `num_predict` |
| `frequency_penalty` | Penalización por repetición | `repeat_penalty` |

**Tiempo estimado:** 30 minutos

## 1. Preparación de la Base Lunar (Instalación)

Antes de conectar con el sistema de comunicaciones, necesitamos instalar las herramientas.

**Requisitos previos:**
- Tener [Ollama](https://ollama.com/) instalado y corriendo (`ollama serve` en una terminal)
- Haber descargado el modelo: `ollama pull gemma:2b`

In [ ]:
# Instalamos la libreria de Python para comunicarnos con Ollama
!pip install ollama -q

In [ ]:
import ollama

def generar_carta(prompt, opciones=None, modelo="gemma4:e2b"):
    """
    Genera texto usando Ollama con las opciones especificadas.

    Args:
        prompt: El texto de instruccion para el modelo.
        opciones: Diccionario con parametros (temperature, top_p, num_predict, repeat_penalty).
        modelo: Nombre del modelo a usar.
    """
    respuesta = ollama.generate(
        model=modelo,
        prompt=prompt,
        options=opciones or {},
        think=False  # Desactiva el razonamiento interno (thinking tokens) que consume num_predict
    )
    print("=" * 60)
    print(f"Parametros usados: {opciones or 'default'}")
    print("=" * 60)
    print(respuesta["response"])
    print(f"\n[Tokens generados: {respuesta.get('eval_count', 'N/A')}]")
    return respuesta["response"]

print("Sistemas de comunicacion listos. Conexion con Ollama establecida.")

## 2. Primera prueba de comunicación

Antes de ajustar parámetros, probemos que la conexión con el modelo funciona. Enviaremos un mensaje simple.

In [ ]:
# Prueba basica de conexion
respuesta = ollama.generate(
    model="gemma4:e2b",
    prompt="Di 'Conexion establecida desde la Base Selene' y nada mas.",
    think=False
)
print(respuesta["response"])

## 3. Temperature: El termómetro emocional

La **temperature** controla qué tan "creativo" o "predecible" es el modelo.

| Valor | Efecto |
|---|---|
| `0.0 - 0.3` | Muy predecible, formal, repetitivo. Como un informe técnico. |
| `0.4 - 0.7` | Equilibrado. Coherente pero con cierta variedad. |
| `0.8 - 1.5` | Muy creativo, impredecible. Puede inventar cosas inesperadas. |

**Escenario:** Van a dictarle al sistema una carta para su familia. Primero con temperature baja (modo "informe de misión") y luego con temperature alta (modo "carta emocional").

In [ ]:
prompt_base = """Eres un astronauta en la Luna. Escribe un parrafo corto (3-4 oraciones)
de una carta para tu familia contandoles como fue tu primer paseo lunar."""

# Temperature BAJA: modo informe tecnico
print("TEMPERATURE = 0.1 (Fria, precisa, predecible)")
print()
carta_fria = generar_carta(prompt_base, opciones={"temperature": 0.1})

In [ ]:
# Temperature ALTA: modo carta emotiva
print("TEMPERATURE = 1.2 (Calida, creativa, impredecible)")
print()
carta_calida = generar_carta(prompt_base, opciones={"temperature": 1.2})

## 4. Top_p: El filtro de vocabulario

**Top_p** (nucleus sampling) controla la diversidad del vocabulario que el modelo puede usar.

- `top_p = 0.1` --> Solo usa las palabras más probables. Vocabulario limitado.
- `top_p = 0.9` --> Considera un rango amplio de palabras. Vocabulario rico.

Piensen en esto: cuando escriben un informe técnico usan pocas palabras (top_p bajo). Cuando escriben poesía, usan todo su vocabulario (top_p alto).

**Escenario:** Van a describir lo que ven desde la ventana de la base lunar.

In [ ]:
prompt_ventana = """Eres un astronauta en una base lunar. Describe en un parrafo (3-4 oraciones)
lo que ves al mirar por la ventana de la base hacia el espacio."""

# Top_p BAJO: vocabulario restringido
print("TOP_P = 0.1 (Vocabulario restringido)")
print()
vista_simple = generar_carta(prompt_ventana, opciones={"top_p": 0.1, "temperature": 0.7})

print("\n" + "~" * 60 + "\n")

# Top_p ALTO: vocabulario amplio
print("TOP_P = 0.95 (Vocabulario amplio)")
print()
vista_rica = generar_carta(prompt_ventana, opciones={"top_p": 0.95, "temperature": 0.7})

## 5. Num_predict (max_tokens): El límite de palabras

**num_predict** controla la longitud máxima de la respuesta (en tokens, donde 1 token es aprox. 3/4 de una palabra).

- `num_predict = 30` --> Respuesta muy corta. Como un telegrama.
- `num_predict = 200` --> Respuesta mediana. Una carta breve.
- `num_predict = 500` --> Respuesta larga. Una carta detallada.

**Escenario:** La ventana de comunicación se está cerrando. Primero solo tienen tiempo para un mensaje de emergencia (corto). Luego la ventana se reabre y pueden enviar un mensaje completo.

In [ ]:
prompt_mensaje = """Eres un astronauta en la Luna. Escribe un mensaje para tu familia
explicandoles que estas bien y contandoles algo interesante del dia."""

# Mensaje CORTO: telegrama espacial
print("NUM_PREDICT = 30 (Telegrama espacial)")
print()
telegrama = generar_carta(prompt_mensaje, opciones={"num_predict": 30, "temperature": 0.7})
print("\n" + "~" * 60 + "\n")

# Mensaje LARGO: carta completa
print("NUM_PREDICT = 200 (Carta completa)")
print()
carta = generar_carta(prompt_mensaje, opciones={"num_predict": 200, "temperature": 0.7})

## 6. Repeat_penalty: El detector de repetición

**repeat_penalty** penaliza al modelo cuando repite las mismas palabras o frases.

- `repeat_penalty = 1.0` --> Sin penalización. El modelo puede repetir libremente.
- `repeat_penalty = 1.3` --> Penalización moderada. Evita repeticiones obvias.
- `repeat_penalty = 1.8` --> Penalización fuerte. Fuerza mucha variedad (puede volverse incoherente).

**Escenario:** Un compañero de tripulación le pide al sistema que escriba un reporte del día a día en la base. Sin penalización, el modelo tiende a repetirse. Veamos la diferencia.

In [ ]:
prompt_reporte = """Eres un astronauta en la Luna. Escribe un reporte de 5 oraciones
describiendo tu rutina diaria en la base lunar. Incluye detalles sobre comida,
experimentos y tiempo libre."""

# Sin penalizacion por repeticion
print("REPEAT_PENALTY = 1.0 (Sin penalizacion)")
print()
reporte_repetitivo = generar_carta(prompt_reporte, opciones={
    "repeat_penalty": 1.0,
    "temperature": 0.7,
    "num_predict": 200
})

print("\n" + "~" * 60 + "\n")

# Con penalizacion moderada
print("REPEAT_PENALTY = 1.5 (Penalizacion moderada)")
print()
reporte_variado = generar_carta(prompt_reporte, opciones={
    "repeat_penalty": 1.5,
    "temperature": 0.7,
    "num_predict": 200
})

## 7. Combinando todo: La carta perfecta

Ahora que conocen cada parámetro, es hora de combinarlos. Aquí hay dos "perfiles" predefinidos:

| Perfil | temperature | top_p | num_predict | repeat_penalty | Uso ideal |
|---|---|---|---|---|---|
| **Informe técnico** | 0.2 | 0.3 | 150 | 1.0 | Reportes formales |
| **Carta personal** | 0.8 | 0.9 | 250 | 1.3 | Mensajes emotivos |

In [ ]:
prompt_carta_final = """Eres un astronauta que lleva 3 meses en la Luna.
Escribe una carta a tu familia. Cuentales como te sientes, que has descubierto,
y cuanto los extranas. Firma la carta con tu nombre inventado."""

# Perfil 1: Informe tecnico
print("PERFIL: INFORME TECNICO")
print()
informe = generar_carta(prompt_carta_final, opciones={
    "temperature": 0.2,
    "top_p": 0.3,
    "num_predict": 150,
    "repeat_penalty": 1.0
})

print("\n" + "=" * 60 + "\n")

# Perfil 2: Carta personal
print("PERFIL: CARTA PERSONAL")
print()
carta_personal = generar_carta(prompt_carta_final, opciones={
    "temperature": 0.8,
    "top_p": 0.9,
    "num_predict": 250,
    "repeat_penalty": 1.3
})

## 8. Misión para casa: Tu propia carta lunar

Ahora es tu turno. Tu tarea es:

### Ejercicio

1. **Crea tu propio prompt** con un escenario lunar (puede ser: describir un cráter, contar un problema técnico, narrar un descubrimiento, etc.)
2. **Experimenta con al menos 3 combinaciones diferentes** de parámetros
3. **Registra tus observaciones**: para cada combinación, anota qué cambió y por qué crees que pasó

In [ ]:
# Tu prompt personalizado
mi_prompt = """Eres un astronauta en la Luna. Acabas de descubrir una cueva subterranea
cerca de la base. Escribe un parrafo contandole a tu familia sobre este descubrimiento."""

# Combinacion 1: Creativa y extensa
print("COMBINACION 1: Creativa y extensa")
print()
resultado_1 = generar_carta(mi_prompt, opciones={
    "temperature": 1.0,
    "top_p": 0.9,
    "num_predict": 200,
    "repeat_penalty": 1.3
})

In [ ]:
# Combinacion 2: Precisa y corta (modo telegrama tecnico)
print("COMBINACION 2: Precisa y corta")
print()
resultado_2 = generar_carta(mi_prompt, opciones={
    "temperature": 0.2,
    "top_p": 0.3,
    "num_predict": 50,
    "repeat_penalty": 1.0
})

In [ ]:
# Combinacion 3: Equilibrada con vocabulario variado
print("COMBINACION 3: Equilibrada con vocabulario variado")
print()
resultado_3 = generar_carta(mi_prompt, opciones={
    "temperature": 0.6,
    "top_p": 0.85,
    "num_predict": 150,
    "repeat_penalty": 1.5
})

### Preguntas guía

- ¿Cuál combinación produjo el texto más "humano"?
- ¿Cuál fue la más útil para un reporte técnico?
- ¿Qué pasa si subes la temperature a 2.0? ¿Y si la bajas a 0.0?
- ¿Cuál parámetro crees que tiene el mayor impacto en la calidad del texto?

### Bonus: Usando `ollama.chat()`

`ollama.chat()` permite usar mensajes con roles (system, user, assistant), similar a como funcionan ChatGPT o Claude. Prueba el siguiente ejemplo:

In [ ]:
# Bonus: usando ollama.chat() con roles de conversacion
respuesta = ollama.chat(
    model="gemma4:e2b",
    messages=[
        {"role": "system", "content": "Eres un astronauta poetico en la Luna."},
        {"role": "user", "content": "Describe el amanecer terrestre visto desde la Luna."}
    ],
    options={"temperature": 0.9, "top_p": 0.85, "num_predict": 200},
    think=False
)
print(respuesta["message"]["content"])

---

**Fin de la transmisión. Base Selene, fuera.**